# GYMTRACE — Experimental Results

Predict gym occupancy (`number_people`) using two models:
1. Linear Regression
2. Random Forest Regressor

Upload your cleaned CSV from the previous activity (`gym_crowdedness_clean.csv`), then run the rest of the notebook.

## 1. Upload cleaned dataset

In [ ]:
from google.colab import files
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

work = Path("/content/gymtrace")
data_dir = work / "data"
proof_dir = work / "proof_ml"
data_dir.mkdir(parents=True, exist_ok=True)
proof_dir.mkdir(parents=True, exist_ok=True)

print("Upload gym_crowdedness_clean.csv")
uploaded = files.upload()

csv_path = data_dir / "gym_crowdedness_clean.csv"
for name, content in uploaded.items():
    if name.lower().endswith(".csv"):
        csv_path.write_bytes(content)
        print("Saved", csv_path)
        break
else:
    raise FileNotFoundError("Please upload a .csv file.")

print("Size MB:", round(csv_path.stat().st_size / 1e6, 2))

## 2. Load data and build features

In [ ]:
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

In [ ]:
features = [
    "timestamp", "day_of_week", "is_weekend", "is_holiday",
    "temperature", "is_start_of_semester", "is_during_semester",
    "month", "hour",
]
target = "number_people"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)

## 3. Train Model A — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)
print("Linear Regression trained.")

## 4. Train Model B — Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=16,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_s, y_train)
pred_rf = rf.predict(X_test_s)
print("Random Forest trained.")

## 5. Evaluate both models

In [ ]:
def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    # MAPE with small epsilon to avoid division by zero
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1))) * 100
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE_%": mape}

results = pd.DataFrame({
    "Linear Regression": metrics(y_test, pred_lr),
    "Random Forest": metrics(y_test, pred_rf),
}).T.round(3)

print("Test-set results")
display(results)

best = results["R2"].idxmax()
print("Best model by R²:", best)

## 6. Charts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results.index))
width = 0.25
ax.bar(x - width, results["MAE"], width, label="MAE")
ax.bar(x, results["RMSE"], width, label="RMSE")
ax.bar(x + width, results["R2"], width, label="R²")
ax.set_xticks(x)
ax.set_xticklabels(results.index)
ax.set_title("Model comparison (test set)")
ax.legend()
plt.tight_layout()
fig.savefig(proof_dir / "01_metrics_comparison.png", dpi=150)
plt.show()
print("Saved 01_metrics_comparison.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_test, pred_lr, alpha=0.2, s=8, color="steelblue")
lims = [0, max(y_test.max(), pred_lr.max())]
axes[0].plot(lims, lims, color="black", linewidth=1)
axes[0].set_title("Linear Regression: actual vs predicted")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")

axes[1].scatter(y_test, pred_rf, alpha=0.2, s=8, color="teal")
lims = [0, max(y_test.max(), pred_rf.max())]
axes[1].plot(lims, lims, color="black", linewidth=1)
axes[1].set_title("Random Forest: actual vs predicted")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

plt.tight_layout()
fig.savefig(proof_dir / "02_actual_vs_predicted.png", dpi=150)
plt.show()
print("Saved 02_actual_vs_predicted.png")

In [ ]:
resid_lr = y_test - pred_lr
resid_rf = y_test - pred_rf

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(resid_lr, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Linear Regression residuals")
axes[0].set_xlabel("Actual - Predicted")
axes[1].hist(resid_rf, bins=40, color="teal", edgecolor="white")
axes[1].set_title("Random Forest residuals")
axes[1].set_xlabel("Actual - Predicted")
plt.tight_layout()
fig.savefig(proof_dir / "03_residuals.png", dpi=150)
plt.show()
print("Saved 03_residuals.png")

In [ ]:
# Feature importance from Random Forest
imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
imp.plot(kind="barh", ax=ax, color="teal")
ax.set_title("Random Forest feature importance")
ax.set_xlabel("Importance")
plt.tight_layout()
fig.savefig(proof_dir / "04_feature_importance.png", dpi=150)
plt.show()
print("Saved 04_feature_importance.png")
display(imp.sort_values(ascending=False).round(4).to_frame("importance"))

## 7. Download proof figures

In [ ]:
import shutil

results.to_csv(proof_dir / "metrics_table.csv")
print("Files saved:")
for f in sorted(proof_dir.glob("*")):
    print(" -", f.name)

zip_path = "/content/gymtrace_experimental_results"
shutil.make_archive(zip_path, "zip", proof_dir)
files.download(zip_path + ".zip")